# Klasifikasi EKG 12 Sadapan dengan Machine Learning vs Deep Learning
**Dataset:** PTB-XL (PhysioNet) — 21.388 EKG 12 sadapan, 10 detik, 100 Hz, dari 18.869 pasien.
**Tugas:** klasifikasi *multi-label* 5 superclass diagnostik:

| Kode | Arti |
|---|---|
| NORM | EKG normal |
| MI | Infark miokard |
| STTC | Perubahan segmen ST / gelombang T |
| CD | Gangguan konduksi (blok cabang berkas, blok AV, dll.) |
| HYP | Hipertrofi (LVH, RVH, pembesaran atrium) |

Satu EKG bisa punya lebih dari satu label (misalnya MI + CD), sehingga tiap kelas diprediksi sebagai probabilitas tersendiri (sigmoid), bukan softmax.

**Pembagian data** mengikuti standar PTB-XL: fold 1–8 latih, fold 9 validasi, fold 10 uji (pasien tidak tumpang tindih antar-fold).

> Jalankan di Google Colab: *Runtime → Change runtime type → GPU (T4)*. Dengan GPU, pelatihan CNN ±2–3 menit; di CPU ±30 menit.

In [ ]:
!pip -q install wfdb
!wget -q -O ptbxl.zip https://physionet-open.s3.amazonaws.com/ptb-xl/ptb-xl-1.0.3.zip
!unzip -q ptbxl.zip '*/records100/*' '*/ptbxl_database.csv' '*/scp_statements.csv' -d raw && rm ptbxl.zip

## 1. Preprocessing
Setiap kode SCP (misalnya `IMI` = *inferior MI*, `LBBB`) dipetakan ke superclass-nya memakai tabel `scp_statements.csv`. Rekaman tanpa label diagnostik dibuang.

In [ ]:
"""Preprocessing PTB-XL (100 Hz) -> numpy arrays + label superclass (multi-label)."""
import ast, glob, os
import numpy as np, pandas as pd, wfdb

ROOT = glob.glob('raw/*')[0] + '/'
df = pd.read_csv(ROOT + 'ptbxl_database.csv', index_col='ecg_id')
df.scp_codes = df.scp_codes.apply(ast.literal_eval)
scp = pd.read_csv(ROOT + 'scp_statements.csv', index_col=0)
scp = scp[scp.diagnostic == 1]

CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

def superclass(codes):
    return list({scp.loc[c, 'diagnostic_class'] for c in codes if c in scp.index})

df['super'] = df.scp_codes.apply(superclass)
df = df[df.super.map(len) > 0]  # buang rekaman tanpa label diagnostik
Y = np.array([[c in s for c in CLASSES] for s in df.super], dtype=np.float32)

X = np.stack([wfdb.rdsamp(ROOT + f)[0] for f in df.filename_lr]).astype(np.float32)  # (N, 1000, 12)
np.savez_compressed('ptbxl100.npz', X=X, Y=Y, fold=df.strat_fold.values,
                    age=df.age.values, sex=df.sex.values, ecg_id=df.index.values)
print(X.shape, Y.sum(0), dict(zip(CLASSES, Y.mean(0).round(3))))


## 2. Baseline: Machine Learning Klasik
Pendekatan "lama": manusia merancang fitur dulu, lalu model belajar dari fitur itu.
Fitur per sadapan: rata-rata, simpangan baku, min/maks, amplitudo puncak-ke-puncak, *skewness*, *kurtosis*, persentil, daya spektral di 4 pita frekuensi; ditambah laju jantung & variabilitas interval RR dari puncak R sadapan II, serta usia dan jenis kelamin. Model: *gradient boosting*, satu model per kelas.

In [ ]:
"""Baseline machine learning klasik: fitur buatan tangan + gradient boosting.
Fitur per sadapan: statistik amplitudo, laju jantung dari puncak R (sadapan II), daya spektral."""
import json, numpy as np
from scipy.signal import find_peaks, welch
from scipy.stats import skew, kurtosis
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
d = np.load('ptbxl100.npz')
X, Y, fold, age, sex = d['X'], d['Y'], d['fold'], d['age'], d['sex']
FS = 100

def feats(x):  # x: (1000, 12)
    f = [x.mean(0), x.std(0), x.min(0), x.max(0), np.ptp(x, 0), skew(x, 0), kurtosis(x, 0),
         np.percentile(x, 5, 0), np.percentile(x, 95, 0), np.abs(np.diff(x, axis=0)).mean(0)]
    fr, p = welch(x, FS, axis=0, nperseg=256)
    for lo, hi in [(0.5, 3), (3, 8), (8, 15), (15, 40)]:
        f.append(np.log(p[(fr >= lo) & (fr < hi)].sum(0) + 1e-8))
    ii = x[:, 1]  # sadapan II untuk puncak R
    pk, _ = find_peaks(ii, distance=30, prominence=0.3 * np.ptp(ii))
    rr = np.diff(pk) / FS if len(pk) > 2 else np.array([1.0, 1.0])
    hr = [60 / rr.mean(), rr.std(), rr.std() / rr.mean(), len(pk)]
    return np.concatenate([np.concatenate(f), hr])

F = np.stack([feats(x) for x in X])
F = np.column_stack([F, age, sex])
print('jumlah fitur:', F.shape[1])
tr, te = fold <= 9, fold == 10
P = np.zeros((te.sum(), 5))
for k in range(5):
    clf = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=0)
    clf.fit(F[tr], Y[tr, k]); P[:, k] = clf.predict_proba(F[te])[:, 1]
np.save('pred_ml_test.npy', P)
res = {'macro_auc_test': roc_auc_score(Y[te], P, average='macro'),
       'per_class': {c: roc_auc_score(Y[te][:, k], P[:, k]) for k, c in enumerate(CLASSES)}}
json.dump(res, open('res_ml.json', 'w'), indent=1); print(json.dumps(res, indent=1))


## 3. Deep Learning: 1D ResNet CNN
Model belajar langsung dari sinyal mentah (12 × 1000 sampel) tanpa fitur buatan tangan.
- **Konvolusi 1D** dengan kernel 7 sampel (70 ms) mengenali pola lokal seperti bentuk QRS, lalu lapisan-lapisan berikutnya menggabungkannya menjadi pola yang makin luas.
- **Koneksi residual** (*skip connection*) memudahkan pelatihan jaringan dalam.
- **Global average + max pooling** merangkum seluruh 10 detik menjadi satu vektor.
- **Augmentasi** (skala amplitudo, noise, geser waktu) mengurangi *overfitting*.
- ±1 juta parameter, optimizer AdamW + jadwal *one-cycle*, 20 epoch.

In [ ]:
"""Deep learning: 1D ResNet CNN untuk klasifikasi 5 superclass PTB-XL (multi-label).
Split standar PTB-XL: fold 1-8 latih, fold 9 validasi, fold 10 uji."""
import time, json, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score

torch.manual_seed(0); np.random.seed(0)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
d = np.load('ptbxl100.npz')
X, Y, fold = d['X'], d['Y'], d['fold']
tr, va, te = fold <= 8, fold == 9, fold == 10
mu, sd = X[tr].mean((0, 1)), X[tr].std((0, 1))           # normalisasi per sadapan
X = ((X - mu) / sd).transpose(0, 2, 1).astype(np.float32)  # (N, 12, 1000)
np.savez('norm.npz', mu=mu, sd=sd)

class Block(nn.Module):
    def __init__(s, cin, cout, stride):
        super().__init__()
        s.c1 = nn.Conv1d(cin, cout, 7, stride, 3, bias=False); s.b1 = nn.BatchNorm1d(cout)
        s.c2 = nn.Conv1d(cout, cout, 7, 1, 3, bias=False); s.b2 = nn.BatchNorm1d(cout)
        s.sk = nn.Sequential() if (cin == cout and stride == 1) else nn.Sequential(
            nn.Conv1d(cin, cout, 1, stride, bias=False), nn.BatchNorm1d(cout))
        s.act = nn.ReLU()
    def forward(s, x):
        return s.act(s.b2(s.c2(s.act(s.b1(s.c1(x))))) + s.sk(x))

class ResNet1D(nn.Module):
    def __init__(s, n_out=5, w=64):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(12, w, 15, 2, 7, bias=False), nn.BatchNorm1d(w), nn.ReLU(), nn.MaxPool1d(3, 2, 1))
        s.layers = nn.Sequential(Block(w, w, 1), Block(w, w, 1), Block(w, 2*w, 2), Block(2*w, 2*w, 1),
                                 Block(2*w, 2*w, 2), Block(2*w, 2*w, 1))
        s.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(4*w, n_out))
    def forward(s, x):
        h = s.layers(s.stem(x))
        h = torch.cat([h.mean(-1), h.amax(-1)], 1)  # global avg + max pooling
        return s.head(h)

def predict(m, x, bs=512):
    m.eval(); out = []; dv = next(m.parameters()).device
    with torch.no_grad():
        for i in range(0, len(x), bs):
            out.append(torch.sigmoid(m(torch.from_numpy(x[i:i+bs]).to(dv))).cpu().numpy())
    return np.concatenate(out)

if True:
    m = ResNet1D().to(dev)
    print('parameter:', sum(p.numel() for p in m.parameters()))
    EPOCHS, BS = 20, 64
    opt = torch.optim.AdamW(m.parameters(), 1e-3, weight_decay=1e-2)
    steps = EPOCHS * int(np.ceil(tr.sum() / BS))
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-3, total_steps=steps)
    lossf = nn.BCEWithLogitsLoss()
    Xtr, Ytr = X[tr], Y[tr]
    best, hist = 0, []
    for ep in range(EPOCHS):
        m.train(); t0 = time.time(); perm = np.random.permutation(len(Xtr)); tl = 0
        for i in range(0, len(perm), BS):
            idx = perm[i:i+BS]
            xb = torch.from_numpy(Xtr[idx]).to(dev)
            # augmentasi ringan: skala amplitudo + noise + geser waktu
            xb = xb * (1 + 0.1*torch.randn(len(idx), 1, 1, device=dev)) + 0.02*torch.randn_like(xb)
            xb = torch.roll(xb, int(np.random.randint(-50, 50)), -1)
            loss = lossf(m(xb), torch.from_numpy(Ytr[idx]).to(dev))
            opt.zero_grad(); loss.backward(); opt.step(); sch.step(); tl += loss.item()*len(idx)
        pv = predict(m, X[va]); auc = roc_auc_score(Y[va], pv, average='macro')
        hist.append({'epoch': ep+1, 'train_loss': tl/len(Xtr), 'val_auc': auc})
        print(f'ep {ep+1:2d} loss {tl/len(Xtr):.4f} val macroAUC {auc:.4f} ({time.time()-t0:.0f}s)', flush=True)
        if auc > best: best = auc; torch.save(m.state_dict(), 'resnet1d_best.pt')
    m.load_state_dict(torch.load('resnet1d_best.pt', map_location=dev))
    pt = predict(m, X[te]); np.save('pred_dl_test.npy', pt)
    res = {'macro_auc_test': roc_auc_score(Y[te], pt, average='macro'),
           'per_class': {c: roc_auc_score(Y[te][:, k], pt[:, k]) for k, c in enumerate(CLASSES)},
           'best_val_auc': best, 'history': hist}
    json.dump(res, open('res_dl.json', 'w'), indent=1)
    print(json.dumps({k: v for k, v in res.items() if k != 'history'}, indent=1))


## 4. Evaluasi & Visualisasi
- **Kurva ROC** per kelas, CNN vs ML klasik.
- **Sensitivitas/spesifisitas** pada ambang optimal (indeks Youden) yang dipilih dari set validasi — bukan dari set uji, agar tidak curang.
- **Peta saliency**: gradien keluaran terhadap input, menunjukkan bagian sinyal yang paling memengaruhi keputusan model (salah satu cara membuka "kotak hitam").

In [ ]:
"""Evaluasi + visualisasi: kurva ROC, perbandingan AUC ML vs DL, contoh EKG + peta saliency."""
import json, numpy as np, torch, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

OUT = 'hasil/'
import os; os.makedirs(OUT, exist_ok=True)
te = fold == 10; Yt = Y[te]
pdl = np.load('pred_dl_test.npy'); pml = np.load('pred_ml_test.npy')
NAMA = {'NORM': 'Normal', 'MI': 'Infark miokard', 'STTC': 'Perubahan ST/T', 'CD': 'Gangguan konduksi', 'HYP': 'Hipertrofi'}
COL = ['#2a7de1', '#e8590c', '#2f9e44', '#9c36b5', '#c2255c']

# 1. Kurva ROC per kelas (DL) + garis putus ML
fig, ax = plt.subplots(1, 5, figsize=(20, 4.2), sharey=True)
for k, c in enumerate(CLASSES):
    for p, ls, lab in [(pdl, '-', 'CNN'), (pml, '--', 'ML klasik')]:
        f, t, _ = roc_curve(Yt[:, k], p[:, k])
        ax[k].plot(f, t, ls, color=COL[k] if ls == '-' else 'gray', lw=2,
                   label=f'{lab} AUC {roc_auc_score(Yt[:, k], p[:, k]):.3f}')
    ax[k].plot([0, 1], [0, 1], ':', color='lightgray'); ax[k].set_title(f'{c} – {NAMA[c]}')
    ax[k].set_xlabel('1 − spesifisitas'); ax[k].legend(loc='lower right', fontsize=9)
ax[0].set_ylabel('Sensitivitas'); plt.tight_layout(); plt.savefig(OUT + 'roc_curves.png', dpi=130); plt.close()

# 2. Sensitivitas & spesifisitas pada ambang yang dipilih dari set validasi (Youden)
m = ResNet1D(); m.load_state_dict(torch.load('resnet1d_best.pt', map_location='cpu')); m.eval()
pv = predict(m, X[fold == 9]); Yv = Y[fold == 9]
rows = []
for k, c in enumerate(CLASSES):
    f, t, th = roc_curve(Yv[:, k], pv[:, k]); thr = th[np.argmax(t - f)]
    yp = pdl[:, k] >= thr; yt = Yt[:, k] == 1
    sens = (yp & yt).sum() / yt.sum(); spec = (~yp & ~yt).sum() / (~yt).sum(); ppv = (yp & yt).sum() / yp.sum()
    rows.append({'kelas': c, 'nama': NAMA[c], 'prevalensi_uji': float(yt.mean()), 'ambang': float(thr),
                 'auc_cnn': roc_auc_score(yt, pdl[:, k]), 'auc_ml': roc_auc_score(yt, pml[:, k]),
                 'sensitivitas': float(sens), 'spesifisitas': float(spec), 'ppv': float(ppv)})
json.dump(rows, open(OUT + 'metrics_test.json', 'w'), indent=1)

# 3. Contoh EKG infark miokard + saliency (gradien input) dari model CNN
idx_te = np.where(te)[0]
cand = [i for j, i in enumerate(idx_te) if Yt[j, 1] == 1 and Yt[j].sum() == 1 and pdl[j, 1] > 0.9]
i = cand[0]
x = torch.from_numpy(X[i:i+1]).requires_grad_(True)
torch.sigmoid(m(x))[0, 1].backward()
sal = x.grad.abs()[0].numpy(); sal = np.convolve(sal.sum(0), np.ones(9)/9, 'same'); sal /= sal.max()
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
raw = np.load('ptbxl100.npz')['X'][i]
tt = np.arange(1000) / 100
fig, ax = plt.subplots(12, 1, figsize=(14, 13), sharex=True)
for l in range(12):
    ax[l].imshow(sal[None, :], aspect='auto', cmap='Reds', alpha=0.45, extent=[0, 10, raw[:, l].min(), raw[:, l].max()])
    ax[l].plot(tt, raw[:, l], 'k', lw=0.8); ax[l].set_ylabel(LEADS[l], rotation=0, labelpad=15); ax[l].set_yticks([])
pr = pdl[np.where(idx_te == i)[0][0]]
ax[0].set_title('EKG uji berlabel MI — prediksi CNN: ' + ', '.join(f'{c} {p:.2f}' for c, p in zip(CLASSES, pr)) +
                '\nArsiran merah = bagian sinyal yang paling memengaruhi keputusan model (saliency)')
ax[-1].set_xlabel('Waktu (detik)'); plt.tight_layout(); plt.savefig(OUT + 'contoh_mi_saliency.png', dpi=110); plt.close()

# 4. Kurva pelatihan
h = json.load(open('res_dl.json'))['history']
fig, a1 = plt.subplots(figsize=(7, 4)); a1.plot([e['epoch'] for e in h], [e['train_loss'] for e in h], 'o-', label='loss latih')
a2 = a1.twinx(); a2.plot([e['epoch'] for e in h], [e['val_auc'] for e in h], 's-', color='#e8590c', label='AUC validasi')
a1.set_xlabel('Epoch'); a1.set_ylabel('BCE loss'); a2.set_ylabel('Macro AUC validasi'); fig.legend(loc='center right')
plt.tight_layout(); plt.savefig(OUT + 'kurva_pelatihan.png', dpi=130); plt.close()
print(json.dumps(rows, indent=1))


## 5. Catatan Interpretasi
- AUC tinggi di data retrospektif **tidak sama** dengan manfaat klinis; perlu validasi eksternal (rumah sakit lain, mesin EKG lain, populasi Indonesia) dan uji klinis seperti EAGLE.
- Nilai prediksi positif (PPV) bergantung pada prevalensi: kelas jarang (HYP) cenderung punya PPV rendah.
- Model ini **alat belajar**, bukan alat diagnosis.